# DJF Box Rainfall Lag-Correlation vs 3-Month Running-Mean Nino-3.4

This notebook builds a DJF rainfall index and correlates it with a **centered 3-month running-mean Nino-3.4** at lags -12 to +12 for:
- **P1**: 1981-2006
- **P2**: 2007-2020

Difference curve is computed as **P2 - P1**.

Target rainfall box (supervisor fixed):
- lon: **95E to 125E**
- lat: **6S to 2N**

Rule applied: if a near-identical stored rectangle already exists in available domain definitions, the notebook uses that exact stored box.


## Lag convention (important)

- DJF year uses: **Dec(y-1), Jan(y), Feb(y)** (example: D1980 + JF1981 = DJF1981).
- For each DJF year **y**, **lag 0 is January of year y**, but the ENSO series is a **centered 3-month running mean**.
- Therefore:
  - **lag 0 = DJF(y)** = mean of **Dec(y-1), Jan(y), Feb(y)**
  - **lag -1 = NDJ(y)** = mean of **Nov(y-1), Dec(y-1), Jan(y)**
  - **lag +1 = JFM(y)** = mean of **Jan(y), Feb(y), Mar(y)**
- More generally, lag **k** uses the centered 3-month running-mean Nino-3.4 value at **Jan(y) + k months**.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, norm
from IPython.display import display

PROJECT_ROOT = Path('/Users/rizzie/TugasAkhir/data_processing')
PATHS_YML = PROJECT_ROOT / 'configs/paths.yml'

TARGET_BOX = {
    'lon_min': 95.0,
    'lon_max': 125.0,
    'lat_min': -6.0,
    'lat_max': 2.0,
}

START_YEAR = 1981
END_YEAR = 2020
P1 = (1981, 2006)
P2 = (2007, 2020)
LAGS = np.arange(-12, 13)

plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25


In [ ]:
def parse_simple_yaml(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f'paths.yml not found: {path}')
    out = {}
    for line in path.read_text(encoding='utf-8').splitlines():
        txt = line.strip()
        if not txt or txt.startswith('#'):
            continue
        if ':' not in txt:
            continue
        k, v = txt.split(':', 1)
        out[k.strip()] = v.strip()
    return out


def first_existing_path(candidates):
    for p in candidates:
        if p is not None and p.exists():
            return p
    raise FileNotFoundError('No existing path found in candidates:\n' + '\n'.join(str(p) for p in candidates if p is not None))


def normalize_box(box: dict) -> dict:
    return {
        'lon_min': float(min(box['lon_min'], box['lon_max'])),
        'lon_max': float(max(box['lon_min'], box['lon_max'])),
        'lat_min': float(min(box['lat_min'], box['lat_max'])),
        'lat_max': float(max(box['lat_min'], box['lat_max'])),
    }


def _iter_geo_points(coords):
    if isinstance(coords, (list, tuple)) and len(coords) >= 2 and all(isinstance(v, (int, float)) for v in coords[:2]):
        yield float(coords[0]), float(coords[1])
        return
    if isinstance(coords, (list, tuple)):
        for sub in coords:
            yield from _iter_geo_points(sub)


def extract_boxes_from_geojson(path: Path):
    data = json.loads(path.read_text(encoding='utf-8'))
    features = data.get('features', []) if isinstance(data, dict) else []
    boxes = []
    for i, feat in enumerate(features, start=1):
        points = list(_iter_geo_points((feat.get('geometry') or {}).get('coordinates', [])))
        if not points:
            continue
        xs = np.array([p[0] for p in points], dtype=float)
        ys = np.array([p[1] for p in points], dtype=float)
        props = feat.get('properties') or {}
        name = props.get('name') or props.get('id') or f'feature_{i}'
        boxes.append(
            {
                'name': str(name),
                'lon_min': float(np.nanmin(xs)),
                'lon_max': float(np.nanmax(xs)),
                'lat_min': float(np.nanmin(ys)),
                'lat_max': float(np.nanmax(ys)),
            }
        )
    return boxes


def resolve_box(target_box: dict, domain_json_candidates, edge_tolerance_deg: float = 1.0):
    target = normalize_box(target_box)
    candidates = []
    for p in domain_json_candidates:
        if p.exists():
            try:
                candidates.extend(extract_boxes_from_geojson(p))
            except Exception as e:
                print(f'Warning: failed to parse {p}: {e}')

    best = None
    best_score = np.inf
    for cand in candidates:
        c = normalize_box(cand)
        diffs = [
            abs(c['lon_min'] - target['lon_min']),
            abs(c['lon_max'] - target['lon_max']),
            abs(c['lat_min'] - target['lat_min']),
            abs(c['lat_max'] - target['lat_max']),
        ]
        if max(diffs) <= edge_tolerance_deg:
            score = float(sum(diffs))
            if score < best_score:
                best = cand
                best_score = score

    if best is not None:
        chosen = normalize_box(best)
        return chosen, f"existing stored box: {best.get('name', 'unknown')}"
    return target, 'supervisor fixed box'


def standardize_rainfall_da(da: xr.DataArray) -> xr.DataArray:
    rename_map = {}
    if 'latitude' in da.dims or 'latitude' in da.coords:
        rename_map['latitude'] = 'lat'
    if 'longitude' in da.dims or 'longitude' in da.coords:
        rename_map['longitude'] = 'lon'
    if 'valid_time' in da.dims or 'valid_time' in da.coords:
        rename_map['valid_time'] = 'time'
    if rename_map:
        da = da.rename(rename_map)

    da = da.assign_coords(time=pd.to_datetime(da['time'].values)).sortby('time')
    da = da.assign_coords(lon=(da['lon'] % 360)).sortby('lon')
    return da


def subset_box(da: xr.DataArray, box: dict) -> xr.DataArray:
    box = normalize_box(box)
    lat0, lat1 = float(da['lat'].values[0]), float(da['lat'].values[-1])
    lon0, lon1 = float(da['lon'].values[0]), float(da['lon'].values[-1])

    lat_slice = slice(box['lat_min'], box['lat_max']) if lat0 <= lat1 else slice(box['lat_max'], box['lat_min'])
    lon_slice = slice(box['lon_min'], box['lon_max']) if lon0 <= lon1 else slice(box['lon_max'], box['lon_min'])
    sub = da.sel(lat=lat_slice, lon=lon_slice)
    if sub.sizes.get('lat', 0) == 0 or sub.sizes.get('lon', 0) == 0:
        raise ValueError(f'Empty subset for box={box}')
    return sub


def area_weighted_monthly_mean(da_box: xr.DataArray) -> pd.Series:
    weights = np.cos(np.deg2rad(da_box['lat']))
    box_mean = da_box.weighted(weights).mean(dim=('lat', 'lon'), skipna=True)
    s = box_mean.to_series()
    s.index = pd.to_datetime(s.index)
    s = s.sort_index().groupby(s.index.to_period('M')).mean()
    s.index = s.index.to_timestamp(how='start')
    s.name = 'rain_box_mm_month'
    return s


def build_djf_rain_index(monthly_rain: pd.Series, start_year: int, end_year: int) -> pd.Series:
    out = {}
    for year in range(start_year, end_year + 1):
        months = [pd.Timestamp(year - 1, 12, 1), pd.Timestamp(year, 1, 1), pd.Timestamp(year, 2, 1)]
        vals = np.array([monthly_rain.get(m, np.nan) for m in months], dtype=float)
        out[year] = np.mean(vals) if np.all(np.isfinite(vals)) else np.nan
    djf = pd.Series(out, name='djf_rain_index')
    djf.index.name = 'djf_year'
    return djf


def load_nino34_monthly(path: Path) -> pd.Series:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    date_col = next((c for c in df.columns if c.lower() == 'date'), None)
    if date_col is None:
        raise ValueError(f'No Date column found in {path}. Columns: {list(df.columns)}')

    value_cols = [c for c in df.columns if c != date_col]
    if not value_cols:
        raise ValueError(f'No Nino3.4 value column found in {path}')

    value_col = value_cols[0]
    tmp = df[[date_col, value_col]].copy()
    tmp['time'] = pd.to_datetime(tmp[date_col], errors='coerce')
    tmp['nino34'] = pd.to_numeric(tmp[value_col], errors='coerce')
    tmp['nino34'] = tmp['nino34'].replace([-99.99, -9999, -9999.0], np.nan)
    tmp = tmp.dropna(subset=['time'])

    tmp['time'] = tmp['time'].dt.to_period('M').dt.to_timestamp(how='start')
    tmp = tmp.groupby('time', as_index=False)['nino34'].mean().sort_values('time')
    s = tmp.set_index('time')['nino34']
    s.name = 'nino34'
    return s


def build_centered_3mo_running_mean(series: pd.Series) -> pd.Series:
    s = series.sort_index().copy()
    s = s.rolling(window=3, center=True, min_periods=3).mean()
    s.name = f'{series.name}_3mo_rm' if series.name else 'nino34_3mo_rm'
    return s


def corr_with_p(x: np.ndarray, y: np.ndarray):
    ok = np.isfinite(x) & np.isfinite(y)
    x_ok = x[ok]
    y_ok = y[ok]
    n = x_ok.size
    if n < 3:
        return np.nan, np.nan, n
    if np.isclose(np.std(x_ok), 0.0) or np.isclose(np.std(y_ok), 0.0):
        return np.nan, np.nan, n
    r, p = pearsonr(x_ok, y_ok)
    return float(r), float(p), int(n)


def lag_corr_table(djf_rain: pd.Series, nino_monthly: pd.Series, years: np.ndarray, lags: np.ndarray) -> pd.DataFrame:
    rows = []
    yvals = djf_rain.reindex(years).to_numpy(dtype=float)
    for lag in lags:
        xvals = []
        for year in years:
            t = pd.Timestamp(int(year), 1, 1) + pd.DateOffset(months=int(lag))
            xvals.append(nino_monthly.get(t, np.nan))
        xvals = np.asarray(xvals, dtype=float)
        r, p, n = corr_with_p(xvals, yvals)
        rows.append({'lag': int(lag), 'r': r, 'p': p, 'n': n})
    return pd.DataFrame(rows)


def fisher_diff_pvalue(r1: float, n1: int, r2: float, n2: int) -> float:
    if not (np.isfinite(r1) and np.isfinite(r2)):
        return np.nan
    if n1 <= 3 or n2 <= 3:
        return np.nan
    z1 = np.arctanh(np.clip(r1, -0.999999, 0.999999))
    z2 = np.arctanh(np.clip(r2, -0.999999, 0.999999))
    se = np.sqrt(1.0 / (n1 - 3) + 1.0 / (n2 - 3))
    if np.isclose(se, 0.0):
        return np.nan
    z = (z2 - z1) / se
    return float(2.0 * (1.0 - norm.cdf(np.abs(z))))


def significance_label(p: float) -> str:
    if not np.isfinite(p):
        return ''
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''


In [ ]:
# 1) Resolve paths from configs/paths.yml
cfg = parse_simple_yaml(PATHS_YML)
climate_data_dir = Path(cfg.get('climate_data_dir', '')) if cfg.get('climate_data_dir') else None
project_data_dir = Path(cfg.get('project_data_dir', '')) if cfg.get('project_data_dir') else None

rain_candidates = [
    (climate_data_dir / 'mswep-monthly/mswep_monthly_combined.nc') if climate_data_dir else None,
    (project_data_dir / 'mswep-monthly/mswep_monthly_combined.nc') if project_data_dir else None,
    PROJECT_ROOT / 'external/ClimateData/mswep-monthly/mswep_monthly_combined.nc',
]

nino_candidates = [
    (climate_data_dir / 'index-monthly/nino34.anom.csv') if climate_data_dir else None,
    (project_data_dir / 'index-monthly/nino34.anom.csv') if project_data_dir else None,
    PROJECT_ROOT / 'external/ClimateData/index-monthly/nino34.anom.csv',
]

domain_json_candidates = [
    (project_data_dir / 'all_data/domain.json') if project_data_dir else None,
    PROJECT_ROOT / 'data/all_data/domain.json',
]
domain_json_candidates = [p for p in domain_json_candidates if p is not None]

RAIN_PATH = first_existing_path(rain_candidates)
NINO34_PATH = first_existing_path(nino_candidates)
BOX, BOX_SOURCE = resolve_box(TARGET_BOX, domain_json_candidates, edge_tolerance_deg=1.0)

# 2) Load monthly MSWEP rainfall and monthly Nino-3.4, then build centered 3-month running mean
ds_rain = xr.open_dataset(RAIN_PATH)
rain_var = 'precipitation' if 'precipitation' in ds_rain.data_vars else list(ds_rain.data_vars)[0]
rain = standardize_rainfall_da(ds_rain[rain_var])
nino_monthly = load_nino34_monthly(NINO34_PATH)
nino_3mo = build_centered_3mo_running_mean(nino_monthly)

# Keep the monthly span needed by DJF 1981..2020 and lags -12..+12 around Jan DJF year.
rain = rain.sel(time=slice('1979-12-01', '2020-02-29'))
nino_3mo = nino_3mo.loc['1980-01-01':'2021-01-01']

# 3) Subset rainfall to box and compute area-weighted monthly box mean
rain_box = subset_box(rain, BOX)
rain_box_monthly = area_weighted_monthly_mean(rain_box)

# 4) Build DJF rainfall index
djf_rain = build_djf_rain_index(rain_box_monthly, START_YEAR, END_YEAR)

print('Rainfall path:', RAIN_PATH)
print('Nino3.4 path:', NINO34_PATH)
print('Box source:', BOX_SOURCE)
print('Box used:', BOX)
print('ENSO series used: centered 3-month running-mean Nino-3.4')


In [ ]:
# 5) Lag-correlation curves for P1 and P2
p1_years = np.arange(P1[0], P1[1] + 1)
p2_years = np.arange(P2[0], P2[1] + 1)

tbl_p1 = lag_corr_table(djf_rain, nino_3mo, p1_years, LAGS).rename(columns={'r': 'r_p1', 'p': 'p_p1', 'n': 'n_p1'})
tbl_p2 = lag_corr_table(djf_rain, nino_3mo, p2_years, LAGS).rename(columns={'r': 'r_p2', 'p': 'p_p2', 'n': 'n_p2'})

result = tbl_p1.merge(tbl_p2, on='lag', how='inner')
result['difference_p2_minus_p1'] = result['r_p2'] - result['r_p1']
result['p_diff'] = [
    fisher_diff_pvalue(r1, n1, r2, n2)
    for r1, n1, r2, n2 in zip(result['r_p1'], result['n_p1'], result['r_p2'], result['n_p2'])
]

result['sig_p1'] = result['p_p1'].map(significance_label)
result['sig_p2'] = result['p_p2'].map(significance_label)
result['sig_diff'] = result['p_diff'].map(significance_label)

table_cols = [
    'lag',
    'r_p1', 'r_p2', 'difference_p2_minus_p1',
    'p_p1', 'p_p2', 'p_diff',
    'sig_p1', 'sig_p2', 'sig_diff',
    'n_p1', 'n_p2',
]

table_out = result[table_cols].copy()
table_out[['r_p1', 'r_p2', 'difference_p2_minus_p1', 'p_p1', 'p_p2', 'p_diff']] = (
    table_out[['r_p1', 'r_p2', 'difference_p2_minus_p1', 'p_p1', 'p_p2', 'p_diff']].round(4)
)

print('Lag table: lag, P1 r, P2 r, difference (P2-P1), and p-values')
display(table_out)

In [ ]:
# 6) Figure 1: P1 and P2 curves
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(result['lag'], result['r_p1'], marker='o', color='tab:blue', label='P1 (1981-2006)')
ax.plot(result['lag'], result['r_p2'], marker='o', color='tab:orange', label='P2 (2007-2020)')
ax.axhline(0.0, color='black', linewidth=1.0, linestyle='--')
ax.axvline(0.0, color='gray', linewidth=1.0, linestyle=':')
ax.set_xlabel('Lag (months; lag 0 = DJF-centered Nino-3.4 window)')
ax.set_ylabel('Pearson r')
ax.set_title('Lag-correlation: DJF rainfall box index vs 3-month running-mean Nino-3.4')
ax.set_xticks(LAGS)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# 7) Figure 2: Difference curve (P2 - P1)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(result['lag'], result['difference_p2_minus_p1'], marker='o', color='tab:purple', label='Difference (P2 - P1)')

sig_mask = result['p_diff'] < 0.05
ax.scatter(
    result.loc[sig_mask, 'lag'],
    result.loc[sig_mask, 'difference_p2_minus_p1'],
    color='red',
    s=45,
    zorder=3,
    label='p_diff < 0.05'
)

ax.axhline(0.0, color='black', linewidth=1.0, linestyle='--')
ax.axvline(0.0, color='gray', linewidth=1.0, linestyle=':')
ax.set_xlabel('Lag (months; lag 0 = DJF-centered Nino-3.4 window)')
ax.set_ylabel('Delta r (P2 - P1)')
ax.set_title('Difference in lag-correlation curves: P2 - P1')
ax.set_xticks(LAGS)
ax.legend()
plt.tight_layout()
plt.show()
